In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
import os
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import seaborn as sns
import pandas as pd
from PIL import Image, UnidentifiedImageError, ImageFilter
import warnings
import time
import random
from huggingface_hub import HfApi, login

warnings.filterwarnings("ignore")

In [ ]:
#SETUP
# =============================================================================
print("=" * 80)
print("🔥 COMPLETE TRAINING + EVALUATION + UPLOAD PIPELINE")
print("=" * 80)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}\n")

In [ ]:
#CUSTOM AUGMENTATION CLASSES
# =============================================================================
class ColorizeStroke(object):
    def __init__(self, p=0.6):
        self.p = p
        self.colors = [
            (0, 0, 255), (0, 255, 0), (255, 165, 0), (255, 20, 147),
            (0, 255, 255), (255, 0, 0), (128, 128, 128), (0, 0, 0)
        ]
    
    def __call__(self, img):
        if random.random() > self.p:
            return img
        if img.mode != 'RGB':
            img = img.convert('RGB')
        color = random.choice(self.colors)
        img_array = np.array(img)
        mask = (img_array.mean(axis=2) < 240)
        colored = img_array.copy()
        for i in range(3):
            colored[:, :, i] = np.where(mask, color[i], colored[:, :, i])
        return Image.fromarray(colored.astype('uint8'))

class AddNoise(object):
    def __init__(self, p=0.3):
        self.p = p
    
    def __call__(self, img):
        if random.random() > self.p:
            return img
        img_array = np.array(img)
        noise = np.random.normal(0, 10, img_array.shape)
        noisy = np.clip(img_array + noise, 0, 255).astype('uint8')
        return Image.fromarray(noisy)

class AddTexture(object):
    def __init__(self, p=0.3):
        self.p = p
    
    def __call__(self, img):
        if random.random() > self.p:
            return img
        return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))

class VaryLineThickness(object):
    def __init__(self, p=0.3):
        self.p = p
    
    def __call__(self, img):
        if random.random() > self.p:
            return img
        if random.random() > 0.5:
            return img.filter(ImageFilter.MinFilter(3))
        else:
            return img.filter(ImageFilter.MaxFilter(3))



In [ ]:

# CONFIGURATION
# =============================================================================
train_val_data_dir = '/kaggle/input/jawa-data'
test_data_dir = '/kaggle/input/dataset-jawa'
train_val_dataset_dir = os.path.join(train_val_data_dir, 'Jawa', 'all_class')
test_dataset_dir = os.path.join(test_data_dir, 'DATASET JAWA')

batch_size = 32
num_classes = 20
train_split = 0.85
num_epochs_phase1 = 10
num_epochs_phase2 = 20
lr_phase1 = 0.001
lr_phase2 = 0.0001

In [ ]:
# TRANSFORMS
# =============================================================================
print("\n📦 Preparing transforms...")

transform_no_aug = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_extreme_aug = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=10),
    ColorizeStroke(p=0.6),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    AddTexture(p=0.4),
    AddNoise(p=0.3),
    VaryLineThickness(p=0.3),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.2),
    transforms.RandomApply([transforms.RandomAdjustSharpness(sharpness_factor=2)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Transforms ready")

In [ ]:
# LOAD DATASETS
# =============================================================================
print("\n📂 Loading datasets...")

def check_image(path):
    try:
        img = Image.open(path)
        img.verify()
        return True
    except:
        return False

dataset_no_aug = datasets.ImageFolder(root=train_val_dataset_dir, transform=transform_no_aug)
dataset_no_aug_val = datasets.ImageFolder(root=train_val_dataset_dir, transform=transform_val_test)
dataset_extreme_aug = datasets.ImageFolder(root=train_val_dataset_dir, transform=transform_extreme_aug)
dataset_extreme_aug_val = datasets.ImageFolder(root=train_val_dataset_dir, transform=transform_val_test)

train_size = int(train_split * len(dataset_no_aug))
val_size = len(dataset_no_aug) - train_size
generator = torch.Generator().manual_seed(42)

train_dataset_no_aug, _ = random_split(dataset_no_aug, [train_size, val_size], generator=generator)
_, val_dataset_no_aug = random_split(dataset_no_aug_val, [train_size, val_size], generator=generator)

train_dataset_extreme, _ = random_split(dataset_extreme_aug, [train_size, val_size], generator=generator)
_, val_dataset_extreme = random_split(dataset_extreme_aug_val, [train_size, val_size], generator=generator)

print(f"✅ Train: {len(train_dataset_no_aug)} | Val: {len(val_dataset_no_aug)}")

test_dataset = datasets.ImageFolder(root=test_dataset_dir, transform=transform_val_test, is_valid_file=check_image)
train_classes = dataset_no_aug.classes
class_names = train_classes

print(f"✅ Test: {len(test_dataset)}")

# DataLoaders
train_loader_no_aug = DataLoader(train_dataset_no_aug, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader_no_aug = DataLoader(val_dataset_no_aug, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
train_loader_extreme = DataLoader(train_dataset_extreme, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader_extreme = DataLoader(val_dataset_extreme, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# Class weights
print("\n⚖️  Calculating class weights...")
train_labels = [dataset_no_aug.targets[idx] for idx in train_dataset_no_aug.indices]
class_counts = Counter(train_labels)
sorted_counts = [class_counts[i] for i in range(num_classes)]
class_weights = [1.0 / count for count in sorted_counts]
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)


In [ ]:
# TRAINING FUNCTION
# =============================================================================
def train_model(model_name, train_loader, val_loader):
    print("\n" + "=" * 80)
    print(f"🚀 TRAINING MODEL: {model_name}")
    print("=" * 80)
    
    model = models.resnet50(pretrained=True)
    num_features = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(num_features, num_classes))
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 
               'epochs': []}
    
    # PHASE 1
    print(f"\n🥶 Phase 1: Training FC layer only...")
    for param in model.parameters():
        param.requires_grad = False
    for param in model.fc.parameters():
        param.requires_grad = True
    
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr_phase1)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=False)
    
    best_val_acc = 0.0
    patience_counter = 0
    epoch_count = 0
    
    for epoch in range(num_epochs_phase1):
        epoch_count += 1
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            train_correct += (predicted == labels).sum().item()
            train_total += labels.size(0)
        
        train_loss /= train_total
        train_acc = train_correct / train_total
        
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)
        
        val_loss /= val_total
        val_acc = val_correct / val_total
        scheduler.step(val_loss)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['epochs'].append(epoch_count)
        
        print(f"Epoch [{epoch+1}/{num_epochs_phase1}] Train: {train_acc*100:.1f}% | Val: {val_acc*100:.1f}%")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= 6:
            print("Early stopping Phase 1")
            break
    
    # PHASE 2
    print(f"\n🔥 Phase 2: Fine-tuning all layers...")
    for param in model.parameters():
        param.requires_grad = True
    
    optimizer = optim.Adam(model.parameters(), lr=lr_phase2)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4, verbose=False)
    
    best_val_acc = 0.0
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(num_epochs_phase2):
        epoch_count += 1
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            train_correct += (predicted == labels).sum().item()
            train_total += labels.size(0)
        
        train_loss /= train_total
        train_acc = train_correct / train_total
        
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)
        
        val_loss /= val_total
        val_acc = val_correct / val_total
        scheduler.step(val_loss)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['epochs'].append(epoch_count)
        
        print(f"Epoch [{epoch+1}/{num_epochs_phase2}] Train: {train_acc*100:.1f}% | Val: {val_acc*100:.1f}%")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            best_model_state = model.state_dict().copy()
            torch.save(best_model_state, f'{model_name}_best.pth')
        else:
            patience_counter += 1
        
        if patience_counter >= 8:
            print("Early stopping Phase 2")
            break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    print(f"\n✅ {model_name} Training Complete! Best Val Acc: {best_val_acc*100:.2f}%")
    
    return model, history, best_val_acc


In [ ]:
# TRAIN BOTH MODELS
# =============================================================================
start_time = time.time()

model_a, history_a, best_val_a = train_model("Model_A_NO_AUG", train_loader_no_aug, val_loader_no_aug)
model_b, history_b, best_val_b = train_model("Model_B_EXTREME_AUG", train_loader_extreme, val_loader_extreme)

total_time = time.time() - start_time
print(f"\n⏱️  Total training time: {total_time/60:.1f} minutes")


In [ ]:
# EVALUATION ON TEST SET
# =============================================================================
print("\n" + "=" * 80)
print("🎯 EVALUATING BOTH MODELS ON TEST SET")
print("=" * 80)

def evaluate_model(model, model_name):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    test_acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return test_acc, np.array(all_preds), np.array(all_labels)

test_acc_a, preds_a, labels_a = evaluate_model(model_a, "Model A")
test_acc_b, preds_b, labels_b = evaluate_model(model_b, "Model B")

print(f"\n📊 TEST SET RESULTS:")
print(f"  Model A (NO AUG)      → Test Acc: {test_acc_a*100:.2f}%")
print(f"  Model B (EXTREME AUG) → Test Acc: {test_acc_b*100:.2f}%")
print(f"  Improvement: {(test_acc_b - test_acc_a)*100:+.2f}%")


In [ ]:
# COMPUTE DETAILED METRICS
# =============================================================================
print("\n" + "=" * 80)
print("📊 COMPUTING DETAILED METRICS")
print("=" * 80)

# Model A
accuracy_a = accuracy_score(labels_a, preds_a)
precision_a = precision_score(labels_a, preds_a, average=None, zero_division=0)
recall_a = recall_score(labels_a, preds_a, average=None, zero_division=0)
f1_a = f1_score(labels_a, preds_a, average=None, zero_division=0)
cm_a = confusion_matrix(labels_a, preds_a)

precision_macro_a = np.mean(precision_a)
recall_macro_a = np.mean(recall_a)
f1_macro_a = np.mean(f1_a)
precision_weighted_a = precision_score(labels_a, preds_a, average='weighted', zero_division=0)
recall_weighted_a = recall_score(labels_a, preds_a, average='weighted', zero_division=0)
f1_weighted_a = f1_score(labels_a, preds_a, average='weighted', zero_division=0)

# Model B
accuracy_b = accuracy_score(labels_b, preds_b)
precision_b = precision_score(labels_b, preds_b, average=None, zero_division=0)
recall_b = recall_score(labels_b, preds_b, average=None, zero_division=0)
f1_b = f1_score(labels_b, preds_b, average=None, zero_division=0)
cm_b = confusion_matrix(labels_b, preds_b)

precision_macro_b = np.mean(precision_b)
recall_macro_b = np.mean(recall_b)
f1_macro_b = np.mean(f1_b)
precision_weighted_b = precision_score(labels_b, preds_b, average='weighted', zero_division=0)
recall_weighted_b = recall_score(labels_b, preds_b, average='weighted', zero_division=0)
f1_weighted_b = f1_score(labels_b, preds_b, average='weighted', zero_division=0)

print(f"\n✅ Model A Metrics:")
print(f"   Accuracy: {accuracy_a*100:.2f}%")
print(f"   Precision (Macro): {precision_macro_a*100:.2f}%")
print(f"   Recall (Macro): {recall_macro_a*100:.2f}%")
print(f"   F1-Score (Macro): {f1_macro_a*100:.2f}%")

print(f"\n✅ Model B Metrics:")
print(f"   Accuracy: {accuracy_b*100:.2f}%")
print(f"   Precision (Macro): {precision_macro_b*100:.2f}%")
print(f"   Recall (Macro): {recall_macro_b*100:.2f}%")
print(f"   F1-Score (Macro): {f1_macro_b*100:.2f}%")

# Classification Reports
print(f"\n📋 Model A Classification Report:")
print(classification_report(labels_a, preds_a, target_names=class_names, digits=4))

print(f"\n📋 Model B Classification Report:")
print(classification_report(labels_b, preds_b, target_names=class_names, digits=4))


In [ ]:
# SAVE METRICS TO CSV
# =============================================================================
print("\n📁 Saving evaluation metrics...")

# Summary comparison
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (Macro)', 'Recall (Macro)', 'F1-Score (Macro)',
               'Precision (Weighted)', 'Recall (Weighted)', 'F1-Score (Weighted)'],
    'Model A (No Aug)': [
        f"{accuracy_a*100:.2f}%", f"{precision_macro_a*100:.2f}%", f"{recall_macro_a*100:.2f}%",
        f"{f1_macro_a*100:.2f}%", f"{precision_weighted_a*100:.2f}%", f"{recall_weighted_a*100:.2f}%",
        f"{f1_weighted_a*100:.2f}%"
    ],
    'Model B (Extreme Aug)': [
        f"{accuracy_b*100:.2f}%", f"{precision_macro_b*100:.2f}%", f"{recall_macro_b*100:.2f}%",
        f"{f1_macro_b*100:.2f}%", f"{precision_weighted_b*100:.2f}%", f"{recall_weighted_b*100:.2f}%",
        f"{f1_weighted_b*100:.2f}%"
    ]
})

comparison_df.to_csv('01_evaluation_metrics_comparison.csv', index=False)
print("✅ Saved: 01_evaluation_metrics_comparison.csv")

# Per-class metrics
support_a = np.bincount(labels_a, minlength=num_classes)
support_b = np.bincount(labels_b, minlength=num_classes)

df_a = pd.DataFrame({
    'Class': class_names,
    'Precision': precision_a,
    'Recall': recall_a,
    'F1-Score': f1_a,
    'Support': support_a
})

df_b = pd.DataFrame({
    'Class': class_names,
    'Precision': precision_b,
    'Recall': recall_b,
    'F1-Score': f1_b,
    'Support': support_b
})

df_a.to_csv('02a_per_class_metrics_model_a.csv', index=False)
df_b.to_csv('02b_per_class_metrics_model_b.csv', index=False)
print("✅ Saved: per-class metrics CSV files")


In [ ]:
# SAVE VISUALIZATIONS
# =============================================================================
print("\n📊 Generating visualizations...")

# 1. Training Loss & Accuracy Curves
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Training Comparison: Model A vs Model B', fontsize=16, fontweight='bold')

# Loss A
axes[0, 0].plot(history_a['epochs'], history_a['train_loss'], label='Train', marker='o', linewidth=2)
axes[0, 0].plot(history_a['epochs'], history_a['val_loss'], label='Val', marker='s', linewidth=2)
axes[0, 0].set_title('Model A: Loss (No Aug)', fontweight='bold', fontsize=12)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Loss B
axes[0, 1].plot(history_b['epochs'], history_b['train_loss'], label='Train', marker='o', linewidth=2)
axes[0, 1].plot(history_b['epochs'], history_b['val_loss'], label='Val', marker='s', linewidth=2)
axes[0, 1].set_title('Model B: Loss (Extreme Aug)', fontweight='bold', fontsize=12)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Accuracy A
axes[1, 0].plot(history_a['epochs'], np.array(history_a['train_acc'])*100, label='Train', marker='o', linewidth=2)
axes[1, 0].plot(history_a['epochs'], np.array(history_a['val_acc'])*100, label='Val', marker='s', linewidth=2)
axes[1, 0].set_title('Model A: Accuracy (No Aug)', fontweight='bold', fontsize=12)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy (%)')
axes[1, 0].set_ylim([0, 105])
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Accuracy B
axes[1, 1].plot(history_b['epochs'], np.array(history_b['train_acc'])*100, label='Train', marker='o', linewidth=2)
axes[1, 1].plot(history_b['epochs'], np.array(history_b['val_acc'])*100, label='Val', marker='s', linewidth=2)
axes[1, 1].set_title('Model B: Accuracy (Extreme Aug)', fontweight='bold', fontsize=12)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy (%)')
axes[1, 1].set_ylim([0, 105])
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('03_training_curves_loss_accuracy.png', dpi=300, bbox_inches='tight')
print("✅ Saved: 03_training_curves_loss_accuracy.png")
plt.close()

# 2. Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(20, 9))

sns.heatmap(cm_a, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, 
            yticklabels=class_names, ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title('Confusion Matrix - Model A (No Aug)', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

sns.heatmap(cm_b, annot=True, fmt='d', cmap='Reds', xticklabels=class_names,
            yticklabels=class_names, ax=axes[1], cbar_kws={'label': 'Count'})
axes[1].set_title('Confusion Matrix - Model B (Extreme Aug)', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig('04_confusion_matrices.png', dpi=300, bbox_inches='tight')
print("✅ Saved: 04_confusion_matrices.png")
plt.close()

# 3. Metrics Comparison Bar Chart
fig, ax = plt.subplots(figsize=(14, 7))

metrics_labels = ['Accuracy', 'Precision\n(Macro)', 'Recall\n(Macro)', 'F1-Score\n(Macro)',
                  'Precision\n(Weighted)', 'Recall\n(Weighted)', 'F1-Score\n(Weighted)']
values_a_list = [accuracy_a*100, precision_macro_a*100, recall_macro_a*100, f1_macro_a*100,
                 precision_weighted_a*100, recall_weighted_a*100, f1_weighted_a*100]
values_b_list = [accuracy_b*100, precision_macro_b*100, recall_macro_b*100, f1_macro_b*100,
                 precision_weighted_b*100, recall_weighted_b*100, f1_weighted_b*100]

x = np.arange(len(metrics_labels))
width = 0.35

bars1 = ax.bar(x - width/2, values_a_list, width, label='Model A (No Aug)', 
               color='skyblue', alpha=0.8, edgecolor='black')
bars2 = ax.bar(x + width/2, values_b_list, width, label='Model B (Extreme Aug)',
               color='salmon', alpha=0.8, edgecolor='black')

ax.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
ax.set_title('Evaluation Metrics Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_labels, fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim([0, 105])
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('05_metrics_comparison_bar.png', dpi=300, bbox_inches='tight')
print("✅ Saved: 05_metrics_comparison_bar.png")
plt.close()

# 4. Per-Class F1-Score Comparison
fig, ax = plt.subplots(figsize=(16, 7))

x = np.arange(len(class_names))
width = 0.35

bars1 = ax.bar(x - width/2, f1_a*100, width, label='Model A (No Aug)',
               color='skyblue', alpha=0.8, edgecolor='black')
bars2 = ax.bar(x + width/2, f1_b*100, width, label='Model B (Extreme Aug)',
               color='salmon', alpha=0.8, edgecolor='black')

ax.set_ylabel('F1-Score (%)', fontsize=12, fontweight='bold')
ax.set_title('Per-Class F1-Score Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim([0, 105])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('06_per_class_f1_comparison.png', dpi=300, bbox_inches='tight')
print("✅ Saved: 06_per_class_f1_comparison.png")
plt.close()

# 5. Val vs Test Accuracy Comparison
fig, ax = plt.subplots(figsize=(10, 6))

metrics = ['Val Accuracy', 'Test Accuracy']
model_a_vals = [best_val_a*100, test_acc_a*100]
model_b_vals = [best_val_b*100, test_acc_b*100]

x = np.arange(len(metrics))
width = 0.35

bars1 = ax.bar(x - width/2, model_a_vals, width, label='Model A (No Aug)',
               color='skyblue', alpha=0.8, edgecolor='black', linewidth=2)
bars2 = ax.bar(x + width/2, model_b_vals, width, label='Model B (Extreme Aug)',
               color='salmon', alpha=0.8, edgecolor='black', linewidth=2)

ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Model Comparison: Validation vs Test Accuracy', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.legend(fontsize=11, loc='lower left')
ax.set_ylim([0, 105])
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('07_val_vs_test_accuracy.png', dpi=300, bbox_inches='tight')
print("✅ Saved: 07_val_vs_test_accuracy.png")
plt.close()


In [ ]:
# UPLOAD TO HUGGING FACE
# =============================================================================
print("\n" + "=" * 80)
print("🚀 UPLOADING MODELS TO HUGGING FACE")
print("=" * 80)

try:
    # Login to Hugging Face
    print("\n🔐 Logging in to Hugging Face...")
    login(token=HF_TOKEN, add_to_git_credential=True)
    
    api = HfApi()
    
    # Upload Model A
    print(f"\n📤 Uploading Model A to {HF_REPO_A}...")
    api.upload_file(
        path_or_fileobj='Model_A_NO_AUG_best.pth',
        path_in_repo='pytorch_model.pth',
        repo_id=HF_REPO_A,
        repo_type='model'
    )
    print(f"✅ Model A uploaded! Download: https://huggingface.co/{HF_REPO_A}/blob/main/pytorch_model.pth")
    
    # Upload Model B
    print(f"\n📤 Uploading Model B to {HF_REPO_B}...")
    api.upload_file(
        path_or_fileobj='Model_B_EXTREME_AUG_best.pth',
        path_in_repo='pytorch_model.pth',
        repo_id=HF_REPO_B,
        repo_type='model'
    )
    print(f"✅ Model B uploaded! Download: https://huggingface.co/{HF_REPO_B}/blob/main/pytorch_model.pth")
    
    # Upload CSV files
    print(f"\n📤 Uploading evaluation metrics...")
    api.upload_file(
        path_or_fileobj='01_evaluation_metrics_comparison.csv',
        path_in_repo='evaluation_metrics.csv',
        repo_id=HF_REPO_A,
        repo_type='model'
    )
    
    # Upload visualization files
    for img_file in ['03_training_curves_loss_accuracy.png', '04_confusion_matrices.png',
                     '05_metrics_comparison_bar.png', '06_per_class_f1_comparison.png',
                     '07_val_vs_test_accuracy.png']:
        api.upload_file(
            path_or_fileobj=img_file,
            path_in_repo=img_file,
            repo_id=HF_REPO_A,
            repo_type='model'
        )
    
    print("\n✅ All files uploaded successfully!")
    
except Exception as e:
    print(f"\n⚠️  Hugging Face upload skipped (make sure HF_TOKEN is set): {e}")
    print("Models saved locally - ready to upload manually")


In [ ]:
# FINAL SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("🏆 FINAL SUMMARY & RESULTS")
print("=" * 80)

print(f"\n📊 ACCURACY COMPARISON:")
print(f"   Model A (No Aug)      - Val: {best_val_a*100:6.2f}% | Test: {test_acc_a*100:6.2f}% | Gap: {(best_val_a-test_acc_a)*100:6.2f}%")
print(f"   Model B (Extreme Aug) - Val: {best_val_b*100:6.2f}% | Test: {test_acc_b*100:6.2f}% | Gap: {(best_val_b-test_acc_b)*100:6.2f}%")
print(f"   Improvement           -                    +{(test_acc_b-test_acc_a)*100:6.2f}%")

print(f"\n🎯 KEY METRICS:")
print(f"   Model A (No Aug):")
print(f"      Accuracy: {accuracy_a*100:.2f}% | Precision (Macro): {precision_macro_a*100:.2f}% | Recall: {recall_macro_a*100:.2f}% | F1: {f1_macro_a*100:.2f}%")
print(f"   Model B (Extreme Aug):")
print(f"      Accuracy: {accuracy_b*100:.2f}% | Precision (Macro): {precision_macro_b*100:.2f}% | Recall: {recall_macro_b*100:.2f}% | F1: {f1_macro_b*100:.2f}%")

if (test_acc_b - test_acc_a) > 0.05:  # > 5% improvement
    print(f"\n🎉 WINNER: Model B (Extreme Augmentation)")
    print(f"   ✅ Superior generalization (smaller Val-Test gap)")
    print(f"   ✅ Better robustness to real-world variations")
    print(f"   ✅ Augmentation strategy validated!")
else:
    print(f"\n📊 Results are comparable - Model selection depends on deployment needs")

print(f"\n📁 OUTPUT FILES SAVED:")
print(f"   ✅ Models:")
print(f"      - Model_A_NO_AUG_best.pth")
print(f"      - Model_B_EXTREME_AUG_best.pth")
print(f"   ✅ Evaluation Metrics:")
print(f"      - 01_evaluation_metrics_comparison.csv")
print(f"      - 02a_per_class_metrics_model_a.csv")
print(f"      - 02b_per_class_metrics_model_b.csv")
print(f"   ✅ Visualizations:")
print(f"      - 03_training_curves_loss_accuracy.png (EPOCH LOSS & ACCURACY)")
print(f"      - 04_confusion_matrices.png")
print(f"      - 05_metrics_comparison_bar.png")
print(f"      - 06_per_class_f1_comparison.png")
print(f"      - 07_val_vs_test_accuracy.png")

if HF_TOKEN != "your_hugging_face_token_here":
    print(f"\n🔗 DOWNLOAD LINKS:")
    print(f"   Model A: https://huggingface.co/{HF_REPO_A}/blob/main/pytorch_model.pth")
    print(f"   Model B: https://huggingface.co/{HF_REPO_B}/blob/main/pytorch_model.pth")
else:
    print(f"\n⚠️  Hugging Face upload requires valid token - upload manually from Kaggle")

print("\n" + "=" * 80)
print("✅ ALL TASKS COMPLETED!")
print("=" * 80)